In [6]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

library(mixOmics)
library(BiocParallel)

Loading required package: MASS

Loading required package: lattice

Loading required package: ggplot2


Loaded mixOmics 6.26.0
Thank you for using mixOmics!
Tutorials: http://mixomics.org
Bookdown vignette: https://mixomicsteam.github.io/Bookdown
Questions, issues: Follow the prompts at http://mixomics.org/contact-us
Cite us:  citation('mixOmics')




In [1]:
DATA_DIR <- "/home/mariya/Data/pallas"

TRAIN_DATA_DIR <- file.path(DATA_DIR, "labeled-diagnostic-bimodal-train")
METH_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.dnam.csv")
GEX_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.gex.csv")
LABELS_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "labels.csv")

TEST_DATA_DIR <- file.path(DATA_DIR, "labeled-diagnostic-bimodal-test")
METH_TEST_FILE <- file.path(TEST_DATA_DIR, "features.dnam.csv")
GEX_TEST_FILE <- file.path(TEST_DATA_DIR, "features.gex.csv")
LABELS_TEST_FILE <- file.path(TEST_DATA_DIR, "labels.csv")

OUTPUT_DIR <- file.path(DATA_DIR, "mixOmics")

In [2]:
source("helpers.R")
train_data <- data(GEX_TRAIN_FILE, METH_TRAIN_FILE, LABELS_TRAIN_FILE)
test_data <- data(GEX_TEST_FILE, METH_TEST_FILE, LABELS_TEST_FILE)

Samples: 827 
GEX features: 5000 
DNAm features: 5000 
Subtypes: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214 AML, ETV6-r AML, KAT6A-r AML, KMT2A-r AML, NPM1alt AML, NUP98-r AML, RUNX1::RUNX1T1 B-ALL, BCR::ABL1 B-ALL, BCR::ABL1-like B-ALL, DUX4-r B-ALL, ETV6::RUNX1 B-ALL, ETV6::RUNX1-like B-ALL, high hyperdiploid B-ALL, iAMP21 B-ALL, KMT2A-r B-ALL, MEF2D-r B-ALL, PAX5alt B-ALL, TCF3::PBX1 B-ALL, ZNF384-r Non-leukemic T-ALL 
Samples: 356 
GEX features: 5000 
DNAm features: 5000 
Subtypes: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214 AML, ETV6-r AML, KAT6A-r AML, KMT2A-r AML, NPM1alt AML, NUP98-r AML, RUNX1::RUNX1T1 B-ALL, BCR::ABL1 B-ALL, BCR::ABL1-like B-ALL, DUX4-r B-ALL, ETV6::RUNX1 B-ALL, ETV6::RUNX1-like B-ALL, high hyperdiploid B-ALL, iAMP21 B-ALL, KMT2A-r B-ALL, MEF2D-r B-ALL, PAX5alt B-ALL, TCF3::PBX1 B-ALL, ZNF384-r Non-leukemic T-ALL 


In [3]:
X_gex_train <- train_data$X_gex
X_dnam_train <- train_data$X_dnam
Y_train     <- train_data$Y

head(X_gex_train)
head(X_dnam_train)
head(Y_train)

,ENSG00000254772,ENSG00000196230,ENSG00000136167,ENSG00000120129,ENSG00000152518,ENSG00000130066,ENSG00000108849,ENSG00000145741,ENSG00000147127,ENSG00000278677,⋯,ENSG00000169750,ENSG00000185909,ENSG00000198049,ENSG00000187003,ENSG00000163352,ENSG00000115350,ENSG00000162931,ENSG00000162614,ENSG00000120332,ENSG00000109089
ALL_1005,12.27591,11.22974,10.214167,11.539675,6.593705,7.270486,-1.1224793,9.414004,-1.122479,11.90285,⋯,2.704500,1.223570,3.869024,-1.122479,-1.122479,4.371454,0.3449763,-0.3603695,-1.1224793,1.3624963
ALL_1019,12.14234,10.60269,10.543023,12.691724,9.851312,10.678990,-1.1224793,8.866561,-1.122479,12.25811,⋯,4.260742,3.194281,2.078454,-1.122479,-1.122479,6.013968,0.6248403,3.5093036,-0.9410244,1.1341167
ALL_1039,12.55317,10.23541,11.136378,12.621307,9.561850,11.659686,-1.1224793,9.288064,-1.122479,12.36581,⋯,2.729294,2.835822,-1.122479,-1.122479,-1.122479,5.221082,-1.1224793,2.6811219,-1.1224793,-0.4911046
ALL_1047,13.20930,12.23359,9.931944,9.745667,10.724379,8.439863,-0.2626047,9.612583,-1.122479,13.44298,⋯,3.711234,2.481087,-1.122479,-1.122479,2.347692,5.628663,-1.1224793,1.3220471,-1.1224793,-0.2988322
ALL_1056,12.87027,10.98678,7.980824,10.820452,8.805872,6.352136,-1.1224793,9.280510,-1.122479,11.06765,⋯,3.319969,1.230886,2.037931,-1.122479,-0.216337,4.241972,2.6970704,1.4042255,-1.1224793,3.1099893
ALL_1060,12.92633,11.58939,8.989700,10.798677,9.713258,8.902986,-1.1224793,9.583214,-1.122479,12.16377,⋯,2.860576,2.028464,-1.122479,-1.122479,2.601327,5.410865,-1.1224793,0.4417425,-0.9123310,-0.7705072


,cg09698465,cg20439288,cg07110356,cg17353893,cg25670583,cg18559579,cg25588389,cg01030534,cg22301418,cg23478547,⋯,cg14293575,cg24563570,cg13931640,cg20722088,cg21531416,cg12103569,cg25970575,cg09187505,cg20165746,cg10725937
ALL_1005,0.6444,0.0292,0.9767,0.9812,0.9345,0.9466,0.0184,0.0606,0.0853,0.9689,⋯,0.6957,0.8926,0.9742,0.9135,0.0456,0.0391,0.4727,0.9609,0.1174,0.7559
ALL_1019,0.3499,0.0218,0.1615,0.9802,0.9477,0.5065,0.0224,0.0402,0.0492,0.9779,⋯,0.1071,0.9407,0.0360,0.0969,0.0562,0.0442,0.4369,0.6563,0.9293,0.4869
ALL_1039,0.9793,0.0216,0.9576,0.9800,0.9485,0.9470,0.6030,0.0354,0.1527,0.9558,⋯,0.3016,0.5240,0.2439,0.1404,0.3471,0.0553,0.2842,0.5419,0.8772,0.7558
ALL_1047,0.9705,0.0298,0.9758,0.9748,0.9202,0.9368,0.0323,0.1199,0.0366,0.9772,⋯,0.1204,0.8708,0.1158,0.2782,0.0909,0.0469,0.6838,0.1312,0.4847,0.9251
ALL_1056,0.1903,0.0186,0.9931,0.2323,0.8878,0.9536,0.0184,0.0350,0.0217,0.9693,⋯,0.2237,0.9510,0.6153,0.9218,0.0642,0.0286,0.9796,0.8294,0.7198,0.9068
ALL_1060,0.9442,0.0241,0.9777,0.9795,0.9413,0.9527,0.0979,0.0329,0.1660,0.9819,⋯,0.1321,0.7779,0.8141,0.6855,0.1132,0.0272,0.6556,0.8378,0.8787,0.9299


[1] B-ALL, MEF2D-r          B-ALL, ZNF384-r         B-ALL, BCR::ABL1       
[4] B-ALL, ETV6::RUNX1-like B-ALL, MEF2D-r          B-ALL, ETV6::RUNX1     
25 Levels: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 ... T-ALL

In [4]:
X_gex_test <- test_data$X_gex
X_dnam_test <- test_data$X_dnam
Y_test     <- test_data$Y

head(X_gex_test)
head(X_dnam_test)
head(Y_test)

,ENSG00000000938,ENSG00000000971,ENSG00000001561,ENSG00000001617,ENSG00000002586,ENSG00000002587,ENSG00000002726,ENSG00000002745,ENSG00000002933,ENSG00000003096,⋯,ENSG00000288636,ENSG00000288640,ENSG00000288645,ENSG00000288683,ENSG00000288701,ENSG00000288722,ENSG00000288825,ENSG00000288859,ENSG00000290292,ENSG00000291237
ALL_1023,7.330034,9.608789,6.910297,5.943203,10.732357,4.162708,0.5487080,0.8494544,3.662242,0.6494256,⋯,0.548708,0.548708,2.973536,2.207838,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708
ALL_1026,8.354619,2.060639,3.836234,3.916173,8.460932,4.397388,0.5487080,0.9115746,1.492451,3.4441405,⋯,0.548708,1.706444,3.546264,3.553404,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708
ALL_106,10.898240,3.017352,8.760607,7.411890,11.765274,2.665644,0.7724149,2.7303208,4.762674,4.3186928,⋯,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708
ALL_1062,3.944970,2.815782,4.542679,2.346339,8.656491,0.548708,0.5487080,0.5487080,0.548708,7.4238706,⋯,0.548708,1.785034,3.545022,3.545767,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708
ALL_1065,13.086489,6.615529,8.444391,1.981466,7.832840,2.864213,0.6296778,2.4165756,5.868387,1.0120847,⋯,3.469955,3.117045,4.761140,1.947142,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708
ALL_1072,5.184384,8.410062,4.994899,2.233427,10.379053,2.631620,3.7463850,3.0026969,4.417025,4.4243722,⋯,2.335893,1.711250,0.548708,5.149184,0.548708,0.548708,0.548708,0.548708,0.548708,0.548708


,cg00001747,cg00001793,cg00003298,cg00010853,cg00011924,cg00025496,cg00027083,cg00031165,cg00034076,cg00037457,⋯,cg27642470,cg27642784,cg27649239,cg27651238,cg27655158,cg27657429,cg27659049,cg27663938,cg27664085,cg27664496
ALL_1023,0.9228,0.1921,0.7217,0.0882,0.7666,0.9442,0.7760,0.7496,0.8323,0.7593,⋯,0.8484,0.8658,0.7331,0.8453,0.7944,0.5270,0.9397,0.8442,0.0981,0.9018
ALL_1026,0.9488,0.0799,0.8415,0.1027,0.3235,0.9059,0.8215,0.9382,0.9295,0.8479,⋯,0.8680,0.9456,0.9487,0.7525,0.9081,0.1688,0.9743,0.8608,0.9514,0.9091
ALL_106,0.8141,0.4507,0.3917,0.0922,0.9766,0.9573,0.6449,0.6700,0.7116,0.7254,⋯,0.7086,0.7852,0.3630,0.9007,0.7320,0.3480,0.8169,0.4202,0.6125,0.6156
ALL_1062,0.9755,0.5852,0.9804,0.0982,0.9596,0.9160,0.4668,0.9457,0.9295,0.9644,⋯,0.9294,0.9452,0.9735,0.9630,0.9694,0.3839,0.9527,0.9421,0.9039,0.2855
ALL_1065,0.0527,0.7998,0.1771,0.1580,0.8650,0.9261,0.5346,0.3716,0.4083,0.0944,⋯,0.3039,0.3178,0.0906,0.5257,0.4256,0.1124,0.5958,0.1296,0.4555,0.5594
ALL_1072,0.7101,0.1159,0.2928,0.0844,0.7829,0.9434,0.7425,0.4667,0.8236,0.5693,⋯,0.6448,0.7252,0.2477,0.9704,0.8445,0.0508,0.9425,0.8173,0.4258,0.7202


[1] B-ALL, ETV6::RUNX1      B-ALL, PAX5alt          B-ALL, ETV6::RUNX1-like
[4] B-ALL, MEF2D-r          B-ALL, TCF3::PBX1       B-ALL, BCR::ABL1-like  
25 Levels: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 ... T-ALL

In [8]:
NCOMP <- 35
KEEP_X <- 500

gex_model <- splsda(X_gex_train, Y_train,
                     ncomp = NCOMP,
                     keepX = rep(KEEP_X, NCOMP))

dnam_model <- splsda(X_dnam_train, Y_train,
                      ncomp = NCOMP,
                      keepX = rep(KEEP_X, NCOMP))

diablo_model <- block.splsda(list(gex = X_gex_train, dnam = X_dnam_train), Y_train,
                          ncomp = NCOMP,
                          keepX = list(gex = rep(KEEP_X, NCOMP),
                                       dnam = rep(KEEP_X, NCOMP)))

Warning message:
“The SGCCA algorithm did not converge”
Warning message:
“The SGCCA algorithm did not converge”
Warning message:
“The SGCCA algorithm did not converge”


In [9]:
saveRDS(gex_model,    file.path(OUTPUT_DIR, "gex_model.rds"))
saveRDS(dnam_model,   file.path(OUTPUT_DIR, "dnam_model.rds"))
saveRDS(diablo_model, file.path(OUTPUT_DIR, "diablo_model.rds"))